# Real-Time Guest Experience Intelligence System

## Problem Statement
Hotels often fail to address guest issues in real time, as feedback is typically collected after checkout. This project aims to analyze guest reviews_df, identify issues, and prioritize them based on severity.

In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
!find /content/drive -name "tripadvisor_hotel_reviews_df.csv"

In [99]:
## Load Dataset
import pandas as pd

reviews_df = pd.read_csv('/content/drive/MyDrive/Projects 2026/Real_Time_Guest_Experience_Intelligence_System/tripadvisor_hotel_reviews.csv')

# Display the first few rows of the loaded DataFrame
print(reviews_df.head())

                                              Review  Rating
0  nice hotel expensive parking got good deal sta...       4
1  ok nothing special charge diamond member hilto...       2
2  nice rooms not 4* experience hotel monaco seat...       3
3  unique, great stay, wonderful time hotel monac...       5
4  great stay great stay, went seahawk game aweso...       5


In [100]:
# !pip install nltk

In [101]:
reviews_df['Review'] = reviews_df['Review'].astype(str)
reviews_df.dropna(inplace=True)

## Sentiment Analysis
Using VADER to classify sentiment of each review.

In [102]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [103]:
sia = SentimentIntensityAnalyzer()
reviews_df['sentiment_score'] = reviews_df['Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [104]:
reviews_df.head(6)

,Review,Rating,sentiment_score
0,nice hotel expensive parking got good deal sta...,4,0.9747
1,ok nothing special charge diamond member hilto...,2,0.9787
2,nice rooms not 4* experience hotel monaco seat...,3,0.9889
3,"unique, great stay, wonderful time hotel monac...",5,0.9912
4,"great stay great stay, went seahawk game aweso...",5,0.9797
5,love monaco staff husband stayed hotel crazy w...,5,0.9870


In [105]:
def get_sentiment_label(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

reviews_df['sentiment'] = reviews_df['sentiment_score'].apply(get_sentiment_label)

In [106]:
reviews_df.head(6)

,Review,Rating,sentiment_score,sentiment
0,nice hotel expensive parking got good deal sta...,4,0.9747,Positive
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive
2,nice rooms not 4* experience hotel monaco seat...,3,0.9889,Positive
3,"unique, great stay, wonderful time hotel monac...",5,0.9912,Positive
4,"great stay great stay, went seahawk game aweso...",5,0.9797,Positive
5,love monaco staff husband stayed hotel crazy w...,5,0.9870,Positive


In [107]:
reviews_df['sentiment'].value_counts()

,count
sentiment,
Positive,18831
Negative,1569
Neutral,91


## Issue Categorization
Classifying reviews into business-relevant categories.

In [108]:
categories = {
    "Cleanliness": ["dirty", "smell", "unclean", "stain"],
    "Staff": ["rude", "helpful", "friendly", "staff"],
    "Food": ["food", "breakfast", "restaurant", "taste"],
    "Maintenance": ["ac", "air conditioning", "broken", "noise"],
    "Check-in": ["checkin", "check-in", "waiting", "delay"]
}

In [109]:
def categorize_review(review):
    review = str(review).lower()

    found_categories = []

    for category, keywords in categories.items():
        for word in keywords:
            if word in review:
                found_categories.append(category)
                break

    if not found_categories:
        return "Other"

    return ", ".join(found_categories)

In [110]:
reviews_df['category'] = reviews_df['Review'].apply(categorize_review)

In [111]:
reviews_df['category'].value_counts().head(6)

,count
category,
"Staff, Food, Maintenance",6391
"Food, Maintenance",2683
"Staff, Maintenance",2596
Maintenance,1554
"Staff, Food",1161
Staff,1051


Explode Categories

In [121]:
reviews_df['category'] = reviews_df['category'].apply(lambda x: x.split(', '))

In [122]:
reviews_df = reviews_df.explode('category')

In [123]:
reviews_df.head(5)

,Review,Rating,sentiment_score,sentiment,category,priority_score
0,nice hotel expensive parking got good deal sta...,4,0.9747,Positive,Other,2
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive,Staff,2
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive,Food,2
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive,Maintenance,2
2,nice rooms not 4* experience hotel monaco seat...,3,0.9889,Positive,Staff,7


In [124]:
#Priority Scoring System

In [125]:
category_weights = {
    "Cleanliness": 5,
    "Maintenance": 5,
    "Staff": 4,
    "Check-in": 3,
    "Food": 3,
    "Other": 2
}

def sentiment_weight(label):
    if label == "Negative":
        return 5
    elif label == "Neutral":
        return 2
    else:
        return 0

urgent_keywords = ["not working", "broken", "worst", "bad", "terrible"]

def urgency_score(review):
    review = review.lower()
    for word in urgent_keywords:
        if word in review:
            return 5
    return 0

def calculate_priority(row):
    return (
        sentiment_weight(row['sentiment']) +
        category_weights.get(row['category'], 2) +
        urgency_score(row['Review'])
    )

reviews_df['priority_score'] = reviews_df.apply(calculate_priority, axis=1)

In [126]:
reviews_df['sentiment'].value_counts()

,count
sentiment,
Positive,44756
Negative,3508
Neutral,194


In [127]:
reviews_df.groupby('category')['priority_score'].mean().sort_values(ascending=False)

,priority_score
category,
Cleanliness,8.316092
Maintenance,6.324579
Staff,5.115665
Check-in,4.803438
Food,4.196506
Other,2.960544


In [128]:
reviews_df.head(5)

,Review,Rating,sentiment_score,sentiment,category,priority_score
0,nice hotel expensive parking got good deal sta...,4,0.9747,Positive,Other,2
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive,Staff,4
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive,Food,3
1,ok nothing special charge diamond member hilto...,2,0.9787,Positive,Maintenance,5
2,nice rooms not 4* experience hotel monaco seat...,3,0.9889,Positive,Staff,9


In [130]:
reviews_df.to_csv("/content/drive/MyDrive/Projects 2026/Real_Time_Guest_Experience_Intelligence_System/hotel_analysis.csv", index=False)



## Key Insights

- Cleanliness issues have the highest severity and should be prioritized for operational improvements.
- Maintenance-related complaints strongly correlate with negative sentiment.
- Real-time prioritization of issues can help hotels improve guest satisfaction during the stay.